# 04 — Tensor hypercontraction (THC)

THC decomposes the two-electron integrals as
$$h_{pqrs} \approx \sum_{\mu\nu} X_{p\mu} X_{q\mu} Z_{\mu\nu} X_{r\nu} X_{s\nu},$$
with $X$ an $N \times M$ interpolation-vector matrix and $Z$ an $M \times M$
core tensor. Block-encoding cost drops from $\mathcal{O}(N^2 \Xi)$ for DF to
$\mathcal{O}(N + M \log M)$ for THC. The THC rank $M$ scales as $\sim cN$ with
$c \in [4, 10]$ for molecular systems (Lee et al., PRX Quantum 2, 030305, 2021).

This notebook uses the analytical Lee/Babbush walk-operator cost. THC lambda is
estimated as a fraction of the DF lambda (compression factor 0.5–0.7 depending
on system size), consistent with reported FeMoCo-class results.

In [1]:
import json
import math
import os
import sys
import numpy as np

sys.path.insert(0, ".")
from config import load_config, thc_compression_for, ensure_dirs
cfg = load_config()
ensure_dirs(cfg)

npz_path = os.path.join(cfg["paths"]["data_dir"], "integrals.npz")
npz = np.load(npz_path)
labels = [a["label"] for a in cfg["active_spaces"]]
all_data = {}
for label in labels:
    all_data[label] = {
        "nelec": int(npz[f"{label}_nelec"]),
        "norb": int(npz[f"{label}_norb"]),
        "e_core": float(npz[f"{label}_e_core"]),
        "h1e_eff": npz[f"{label}_h1e_eff"],
        "h2e_phys": npz[f"{label}_h2e_phys"],
    }
epsilon = cfg["qpe"]["epsilon_ha"]
thc_factor = cfg["thc"]["thc_factor"]

## THC resource estimator

In [2]:
def thc_resource_estimate(h1e, h2e, norb, epsilon, thc_factor, cfg):
    """Analytical THC cost, Lee et al. PRX Quantum 2, 030305 (2021)."""
    N = norb
    M = thc_factor * N

    # DF eigenspectrum to anchor the lambda estimate
    h2e_flat = h2e.reshape(N**2, N**2)
    h2e_sym = 0.5 * (h2e_flat + h2e_flat.T)
    eigvals = np.linalg.eigvalsh(h2e_sym)
    eigvals = eigvals[np.abs(eigvals) > 1e-6]

    lambda_1e = np.sum(np.abs(h1e))
    lambda_2e_df = np.sum(np.abs(eigvals))

    compression = thc_compression_for(N, cfg)
    lambda_2e_thc = lambda_2e_df * compression
    lambda_thc = lambda_1e + lambda_2e_thc

    prec = int(np.ceil(np.log2(lambda_thc / epsilon))) + 1
    qpe_rounds = 2 ** prec

    log2_M = max(1, math.ceil(math.log2(M)))

    # Toffoli per walk step (Lee et al., Eq. C26-C30)
    toff_prepare = M * log2_M
    toff_select = 2 * N
    toff_reflect = log2_M
    toff_per_walk = 2 * (toff_prepare + toff_select + toff_reflect)
    t_per_walk = toff_per_walk * 4

    t_total = qpe_rounds * t_per_walk

    ancilla = log2_M + math.ceil(math.log2(N)) + N
    logical_qubits = 2 * N + ancilla + prec + 1

    return {
        "norb": N,
        "thc_rank_M": M,
        "lambda_1e": float(lambda_1e),
        "lambda_2e_df": float(lambda_2e_df),
        "lambda_2e_thc": float(lambda_2e_thc),
        "lambda_total": float(lambda_thc),
        "compression": compression,
        "qpe_bits": int(prec),
        "qpe_rounds": int(qpe_rounds),
        "t_per_walk": int(t_per_walk),
        "t_total": int(t_total),
        "logical_qubits": int(logical_qubits),
    }

## Run THC estimation across all active spaces

In [3]:
thc_results = {}
for label in labels:
    d = all_data[label]
    thc = thc_resource_estimate(d["h1e_eff"], d["h2e_phys"], d["norb"], epsilon, thc_factor, cfg)
    thc["active_space"] = f"({d['nelec']}e, {d['norb']}o)"
    thc_results[label] = thc

    print(f"({d['nelec']}e,{d['norb']}o): M={thc['thc_rank_M']:>3}  "
          f"lambda={thc['lambda_total']:>7.2f}  "
          f"QPE={thc['qpe_bits']:>2}  T={thc['t_total']:>14,}  "
          f"Q_L={thc['logical_qubits']}")

(4e,4o): M= 24  lambda=   6.71  QPE=14  T=    17,432,576  Q_L=34
(8e,8o): M= 48  lambda=  28.46  QPE=16  T=   162,529,280  Q_L=50
(12e,12o): M= 72  lambda=  66.02  QPE=17  T=   560,988,160  Q_L=65
(16e,16o): M= 96  lambda= 112.44  QPE=18  T= 1,491,075,072  Q_L=78
(20e,20o): M=120  lambda= 181.41  QPE=18  T= 1,860,173,824  Q_L=91
(24e,24o): M=144  lambda= 268.59  QPE=19  T= 5,066,719,232  Q_L=105


## Save results

In [4]:
out_path = os.path.join(cfg["paths"]["data_dir"], "thc_results.json")
with open(out_path, "w") as f:
    json.dump(thc_results, f, indent=2)
print(f"saved {out_path}")

saved data/thc_results.json
